# COMP 7640: Ecommerce System - Order Manager part 

This contains database configuration and order related functional testing

In [1]:
import pymysql

# 1. Database Configuration
DB_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': '126416', 
    'database': 'comp7640_ecommerce',
    'charset': 'utf8mb4'
}

In [2]:
# 2. OrderManager Class Implementation

class OrderManager:
    def __init__(self, config):
        self.config = config

    def get_connection(self):
        """Establish database connection"""
        return pymysql.connect(
            host=self.config['host'],
            user=self.config['user'],
            password=self.config['password'],
            database=self.config['database'],
            charset=self.config['charset'],
            cursorclass=pymysql.cursors.DictCursor
        )

    def cancel_order(self, order_id):
        """
        Cancel an entire order (Cancellation of the entire order before it enters the shipping process)
        """
        conn = self.get_connection()
        try:
            with conn.cursor() as cursor:
                # 1. Check if the order status allows cancellation
                cursor.execute("SELECT status FROM orders WHERE order_id = %s", (order_id,))
                result = cursor.fetchone()
                if not result:
                    return False, f"Order {order_id} not found."
                
                if result['status'] != 'PENDING':
                    return False, f"Order {order_id} cannot be cancelled (current status: {result['status']}). Only PENDING orders can be cancelled."

                # 2. Update order status to CANCELLED
                cursor.execute("UPDATE orders SET status = 'CANCELLED' WHERE order_id = %s", (order_id,))
                
                conn.commit()
                return True, f"Order {order_id} has been successfully cancelled."
        except Exception as e:
            conn.rollback()
            return False, f"Error cancelling order: {str(e)}"
        finally:
            conn.close()

    def remove_item_from_order(self, order_id, product_id):
        """
        Remove a specific product from an order (Removal of specific products)
        """
        conn = self.get_connection()
        try:
            with conn.cursor() as cursor:
                # 1. Check if the order status allows modification
                cursor.execute("SELECT status FROM orders WHERE order_id = %s", (order_id,))
                order_result = cursor.fetchone()
                if not order_result:
                    return False, f"Order {order_id} not found."
                
                if order_result['status'] != 'PENDING':
                    return False, f"Order cannot be modified (current status: {order_result['status']})."

                # 2. Retrieve product information
                cursor.execute("""
                    SELECT oi.quantity, oi.price_at_purchase, p.vendor_id 
                    FROM order_items oi
                    JOIN products p ON oi.product_id = p.product_id
                    WHERE oi.order_id = %s AND oi.product_id = %s
                """, (order_id, product_id))
                item_info = cursor.fetchone()
                if not item_info:
                    return False, f"Product {product_id} not found in Order {order_id}."

                item_total_price = item_info['quantity'] * item_info['price_at_purchase']
                vendor_id = item_info['vendor_id']

                # 3. Delete order item
                cursor.execute("DELETE FROM order_items WHERE order_id = %s AND product_id = %s", (order_id, product_id))

                # 4. Update order total price
                cursor.execute("UPDATE orders SET total_price = total_price - %s WHERE order_id = %s", (item_total_price, order_id))

                # 5. Update or delete corresponding transaction amount (to prevent CHECK (amount > 0) constraint violation)
                cursor.execute("""
                    SELECT amount FROM transactions 
                    WHERE order_id = %s AND vendor_id = %s
                """, (order_id, vendor_id))
                trans_result = cursor.fetchone()

                if trans_result:
                    new_amount = trans_result['amount'] - item_total_price
                    if new_amount <= 0:
                        # If amount reaches 0, delete the transaction record directly
                        cursor.execute("DELETE FROM transactions WHERE order_id = %s AND vendor_id = %s", (order_id, vendor_id))
                    else:
                        # Update only if the amount remains greater than 0
                        cursor.execute("""
                            UPDATE transactions 
                            SET amount = %s 
                            WHERE order_id = %s AND vendor_id = %s
                        """, (new_amount, order_id, vendor_id))

                # Automatically cancel empty orders
                cursor.execute("SELECT COUNT(*) as count FROM order_items WHERE order_id = %s", (order_id,))
                count_result = cursor.fetchone()
                if count_result['count'] == 0:
                    cursor.execute("UPDATE orders SET status = 'CANCELLED', total_price = 0 WHERE order_id = %s", (order_id,))

                conn.commit()
                return True, f"Product {product_id} has been removed from Order {order_id}."
        except Exception as e:
            conn.rollback()
            return False, f"Error modifying order: {str(e)}"
        finally:
            conn.close()

In [3]:
# 3. Functional Testing

manager = OrderManager(DB_CONFIG)

print("--- Starting Order Modification Logic Tests ---\n")

# Scenario 1: Successfully remove a product from a multi-vendor order
# Order 1 originally contains Product 1 (iPhone) and Product 3 (PS5)
print("[Test 1] Attempting to remove Product 3 (PS5) from Order 1...")
success, msg = manager.remove_item_from_order(1, 3)
print(f"Result: {msg}")

# Scenario 2: Successfully cancel a PENDING order
# Order 3 status is PENDING
print("\n[Test 2] Attempting to cancel Order 3 (PENDING status)...")
success, msg = manager.cancel_order(3)
print(f"Result: {msg}")

# Scenario 3: Intercept modification for non-PENDING status (Expected to fail)
# Order 2 status is SHIPPED
print("\n[Test 3] Attempting to cancel Order 2 (SHIPPED status)...")
success, msg = manager.cancel_order(2)
print(f"Result: {msg}")

# Scenario 4: Remove a product that does not exist in the order (Expected to fail)
print("\n[Test 4] Attempting to remove non-existent Product 2 from Order 1...")
success, msg = manager.remove_item_from_order(1, 2)
print(f"Result: {msg}")

--- Starting Order Modification Logic Tests ---

[Test 1] Attempting to remove Product 3 (PS5) from Order 1...
Result: Product 3 not found in Order 1.

[Test 2] Attempting to cancel Order 3 (PENDING status)...
Result: Order 3 cannot be cancelled (current status: CANCELLED). Only PENDING orders can be cancelled.

[Test 3] Attempting to cancel Order 2 (SHIPPED status)...
Result: Order 2 cannot be cancelled (current status: SHIPPED). Only PENDING orders can be cancelled.

[Test 4] Attempting to remove non-existent Product 2 from Order 1...
Result: Product 2 not found in Order 1.
